# RL for Optimized Trade Execution — ArXivist Reproduction Notebook
**Paper:** Nevmyvaka, Feng, Kearns — ICML 2006

> Data note: uses synthetic order book data. See `data/README_data.md` for real INET/LOBSTER options.

## Paper in Brief
- **Goal**: Sell V shares within H minutes, minimizing trading cost (bps vs mid-spread)
- **Method**: Tabular backward-induction Q-learning. State = (t, i, market_vars)
- **Key insight**: Market vars evolve independently of actions → O(T×I×L) training, independent of R
- **Results**: 27–50%+ improvement over optimized Submit-and-Leave baseline


In [ ]:
import sys, os, subprocess
repo_root = os.path.abspath('..')
sys.path.insert(0, os.path.join(repo_root, 'src'))
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', repo_root, '-q'], capture_output=True, text=True)
print('Install error:', r.stderr) if r.returncode != 0 else print('Package installed')
import numpy as np
print(f'NumPy {np.__version__} — tabular RL, no GPU needed')


## Q-Learning Update Rule (Section 3)
$$c(x,a) = \\frac{n}{n+1}c(x,a) + \\frac{1}{n+1}\\bigl[c_{im}(x,a)+\\arg\\min_p c(y,p)\\bigr]$$
Training goes **backward** in time from t=T to t=0. At t=T the action is forced (market order). Each previous timestep uses the already-computed future costs.


In [ ]:
# Section 2: Recreate NVDA order book from Figure 1
from rl_trade_execution.env.order_book import OrderBookSnapshot, PriceLevel

snap = OrderBookSnapshot(
    timestamp=0,
    bids=[PriceLevel(27.19,100), PriceLevel(27.18,9), PriceLevel(27.13,109), PriceLevel(27.04,1843)],
    asks=[PriceLevel(27.22,713), PriceLevel(27.27,1000), PriceLevel(27.27,640), PriceLevel(27.43,500)]
)
print(f'bid={snap.bid():.2f}  ask={snap.ask():.2f}  mid={snap.mid():.4f}  spread={snap.spread():.4f}')
filled, avg, rem = snap.simulate_sell_execution(500, snap.ask())
print(f'Sell 500 @ ask: filled={filled}, avg=${avg:.4f}, remaining={rem}')
print(f'Market order cost (1000 shares): {snap.market_order_cost_bps(1000, "sell"):.2f} bps')


In [ ]:
# Generate synthetic data and fit market features (Section 4.2)
from rl_trade_execution.env.market_features import MarketFeatureExtractor
from rl_trade_execution.data.loader import SyntheticOrderBookGenerator

gen = SyntheticOrderBookGenerator(seed=42)
train_eps = gen.generate_episodes(200, T=4)
test_eps  = gen.generate_episodes(100, T=4)
train_snaps = [s for ep in train_eps for s in ep.snapshots]

feat = MarketFeatureExtractor(['bid_ask_spread', 'immediate_market_order_cost'], n_bins=3)
feat.fit(train_snaps, volume=10000, side='sell')
s = train_snaps[20]
f = feat.extract(s, 10000, 'sell')
print(f'Features: {dict(zip(feat.feature_names, f))}  (0=low, 1=med, 2=high)')
print(f'State dims: {feat.state_dims}')


In [ ]:
# Q-table update rule demo
from rl_trade_execution.agent.q_table import QTable

qt = QTable(n_states=5*5*3*3, n_actions=21)
print(f'Q-table: {qt}  |  Memory: {qt.costs.nbytes/1024:.1f} KB')

qt.update(state_idx=10, action_idx=5, immediate_cost=8.5, best_future_cost=12.3)
print(f'After 1st visit: c(10,5)={qt.get(10,5):.2f} bps  (expected {8.5+12.3:.2f})')
qt.update(state_idx=10, action_idx=5, immediate_cost=7.0, best_future_cost=11.0)
print(f'After 2nd visit: c(10,5)={qt.get(10,5):.2f} bps  (avg of 20.8, 18.0 = {(20.8+18.0)/2:.2f})')


In [ ]:
# Full mini training run — backward induction on 200 synthetic episodes
import time
from rl_trade_execution.utils.config import ExperimentConfig, set_seed
from rl_trade_execution.env.market_env import TradeExecutionEnv
from rl_trade_execution.training.trainer import BackwardInductionTrainer

set_seed(42)
cfg = ExperimentConfig(
    stock='SYN', V=10000, H_minutes=2, side='sell',
    T=4, I=4, L=21, action_min=-6, action_max=14,
    market_variables=['bid_ask_spread'], n_bins_market=3, seed=42
)
env = TradeExecutionEnv(cfg, feat)
print(f'State space: {env.total_states:,} states')
print(f'Data passes required: T*I*L = {cfg.T}*{cfg.I}*{cfg.L} = {cfg.T*cfg.I*cfg.L} (independent of R)')

t0 = time.time()
trainer = BackwardInductionTrainer(cfg, env)
q = trainer.train([ep.snapshots for ep in train_eps], log_every=500000)
print(f'Done in {time.time()-t0:.1f}s  |  Coverage: {q.coverage():.2%}')


In [ ]:
# Evaluate RL policy vs Submit-and-Leave baseline
from rl_trade_execution.agent.policy import OptimalPolicy
from rl_trade_execution.baselines.submit_and_leave import SubmitAndLeavePolicy
from rl_trade_execution.evaluation.metrics import ExecutionMetrics

policy = OptimalPolicy.from_q_table(q, cfg)
sl = SubmitAndLeavePolicy.optimize(cfg, [ep.snapshots for ep in train_eps[:50]])

rl_costs, sl_costs = [], []
for ep in test_eps:
    env.reset(ep.snapshots)
    state = env.encode_state(0, cfg.I, feat.extract(ep.snapshots[0], cfg.V, cfg.side))
    c = 0.0
    for _ in range(cfg.T):
        state, cost, done, _ = env.step(policy.act(state))
        c += cost
        if done: break
    rl_costs.append(c)
    sl_costs.append(sl._run_episode(ep.snapshots))

r = ExecutionMetrics.compare_policies(rl_costs, sl_costs, 'RL', 'S&L')
print(r['summary'])
print("Paper benchmark (real INET data): 27.16%-35.50% improvement (private vars only)")


In [ ]:
# Reproduce Figure 4: policy heatmap — aggressive when low time + high inventory
import matplotlib.pyplot as plt
import numpy as np

T, I = cfg.T, cfg.I
grid = np.zeros((I+1, T+1))
neutral = [1] * len(cfg.market_variables)
for t in range(T+1):
    for i in range(I+1):
        idx = env.encode_state(t, i, neutral)
        grid[i, t] = cfg.index_to_action(q.best_action(idx))

fig, ax = plt.subplots(figsize=(8,5))
im = ax.imshow(grid, aspect='auto', cmap='RdYlGn', origin='lower', vmin=-6, vmax=14)
plt.colorbar(im, ax=ax, label='Action (price offset from ask); higher = more aggressive')
ax.set_xlabel('Time Remaining (t)', fontsize=12)
ax.set_ylabel('Inventory Remaining (i units)', fontsize=12)
ax.set_title('Learned Policy (Figure 4 equivalent)\nLow time + high inventory = aggressive pricing', fontsize=11)
ax.set_xticks(range(T+1)); ax.set_yticks(range(I+1))
plt.tight_layout(); plt.savefig('policy_heatmap.png', dpi=100, bbox_inches='tight'); plt.show()


In [ ]:
# Reproduce Figure 5: Q-value curves vs action
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
actions = [cfg.index_to_action(a) for a in range(cfg.L)]

for ax, (fixed_var, fixed_val, vary_var, vary_range) in zip(axes, [
    ('t', 1, 'i', range(1, I+1)),
    ('i', I, 't', range(1, T+1))
]):
    for val in vary_range:
        if fixed_var == 't':
            state = env.encode_state(fixed_val, val, neutral)
            label = f'i={val}'
        else:
            state = env.encode_state(val, fixed_val, neutral)
            label = f't={val}'
        costs = [q.get(state, a) if q.get(state, a) < 1e5 else float('nan') for a in range(cfg.L)]
        ax.plot(actions, costs, marker='o', markersize=3, label=label)
    ax.set_xlabel('Action (price offset)'); ax.set_ylabel('Expected Cost (bps)')
    ax.set_title(f'Fix {fixed_var}={fixed_val}, vary {vary_var}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Q-Value Curves (Figure 5 equivalent)\nMinimum of each curve = optimal action', fontsize=12)
plt.tight_layout(); plt.savefig('qvalue_curves.png', dpi=100, bbox_inches='tight'); plt.show()


## Paper's Reported Numbers (Table 1 + Section 4.1)
For reference when comparing against real INET data results:

| Experiment | Result |
|-----------|--------|
| RL (T=4,I=4) vs S&L | +27.16% improvement |
| RL (T=8,I=8) vs S&L | +35.50% improvement |
| + bid-ask spread | +7.97% over private-only |
| + immediate market order cost | +4.26% over private-only |
| + spread + imm_cost + signed_vol | +12.85% over private-only |
| RL best config vs S&L | **≥50% improvement** |

## Next Steps
1. Obtain real NASDAQ order book data — see `data/README_data.md` (LOBSTER, TotalView-ITCH)
2. `python train.py --config configs/config.yaml`
3. `python evaluate.py --policy models/policy_final.pkl`
